In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformer_lens import HookedTransformer
import torch

import json
import time
import os

from src.neuron_texts import find_truncated_texts

In [3]:
filename = "2026-06-15_19-45-11_gpt2-small-random_test"
is_random = "-random" # otherwise, ""
train = False

with open(f'../experiment_data/2_max_activating_texts/{filename}.json') as f:
    max_text_data = json.load(f)

In [4]:
!ls ../experiment_data/2_max_activating_texts/

2026-06-11_01-13-19_gpt2-small_train.json
2026-06-11_01-23-00_gpt2-small_test.json
2026-06-15_19-24-08_gpt2-small-random_train.json
2026-06-15_19-45-11_gpt2-small-random_test.json


In [5]:
# Parameters
parameters = max_text_data['parameters']
model_name = parameters['model_name']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [7]:
model = HookedTransformer.from_pretrained(
    model_name,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    # refactor_factored_attn_matrices=True,
    device=device,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [8]:
import pickle
dataset_text_test_path = "../experiment_data/text_dict_test.pkl"
dataset_text_train_path = "../experiment_data/text_dict_train.pkl"
if train:
    with open(dataset_text_train_path, 'rb') as f:
        dataset = pickle.load(f)
        
    print("Train loaded!")

else:
    with open(dataset_text_test_path, 'rb') as f:
        dataset = pickle.load(f)

    # removing samples that were also in the training dataset
    dataset.pop(9272) 
    dataset.pop(7138)
    print("Test loaded!")

Test loaded!


In [9]:
num_samples = 20
act_ratio=0.8
neuron_max_acts=max_text_data['neuron_max_acts']

In [10]:
with torch.no_grad():
    neuron_20_examples = find_truncated_texts(
        model=model,
        neuron_max_acts=max_text_data['neuron_max_acts'],
        dataset=dataset,
        device=device,
        num_samples=20,
        act_ratio=0.8,
    )

output = {
    'parameters': parameters,
    'neuron_to_trunc_data': neuron_20_examples,
    'prior_filename': filename,
}


train_string = "_train" if train else "_test"

timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}{is_random}{train_string}.json"

folder_path = "../experiment_data/3_trimmed_texts"
filepath = os.path.join(folder_path, new_filename)
if os.path.isfile(filepath):
    raise Exception("File already exists!")
os.makedirs(folder_path, exist_ok=True)

with open(filepath, 'w') as f:
    json.dump(output, f)

  0%|          | 0/60 [00:00<?, ?it/s]

20it [00:04,  4.07it/s]
20it [00:04,  4.79it/s]00:04<04:51,  4.93s/it]
20it [00:04,  4.34it/s]00:09<04:20,  4.50s/it]
20it [00:04,  4.21it/s]00:13<04:19,  4.55s/it]
20it [00:04,  4.32it/s]00:18<04:19,  4.64s/it]
20it [00:04,  4.28it/s]00:23<04:15,  4.64s/it]
20it [00:04,  4.46it/s]00:27<04:11,  4.66s/it]
20it [00:04,  4.38it/s]00:32<04:04,  4.60s/it]
20it [00:04,  4.18it/s]00:36<03:58,  4.60s/it]
20it [00:04,  4.86it/s]00:41<03:57,  4.66s/it]
20it [00:04,  4.33it/s][00:45<03:44,  4.49s/it]
20it [00:04,  4.55it/s][00:50<03:42,  4.54s/it]
20it [00:05,  3.96it/s][00:54<03:35,  4.50s/it]
20it [00:04,  4.58it/s][00:59<03:39,  4.67s/it]
20it [00:04,  4.25it/s][01:04<03:30,  4.58s/it]
20it [00:04,  4.54it/s][01:09<03:28,  4.63s/it]
20it [00:04,  4.95it/s][01:13<03:20,  4.56s/it]
20it [00:04,  4.18it/s][01:17<03:09,  4.41s/it]
20it [00:04,  4.36it/s][01:22<03:10,  4.53s/it]
20it [00:04,  4.54it/s][01:26<03:06,  4.55s/it]
20it [00:04,  4.58it/s][01:31<03:00,  4.51s/it]
20it [00:04,  4.58it/s][0

In [11]:
output = {
    'parameters': parameters,
    'neuron_to_trunc_data': neuron_20_examples,
    'prior_filename': filename,
}


train_string = "_train" if train else "_test"
# Save json to ../experiment_data/2_max_activating_texts
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}{is_random}{train_string}.json"
folder_path = "../experiment_data/3_trimmed_texts"
filepath = os.path.join(folder_path, new_filename)

if os.path.isfile(filepath):
    raise Exception("File already exists!")
    
os.makedirs(folder_path, exist_ok=True)

with open(filepath, 'w') as f:
    json.dump(output, f)